In [12]:
import pandas as pd 
import numpy as np
import random

In [13]:
df= pd.read_csv('clean_data.csv')

In [7]:
print(df.loc[10, 'product_image'])

[{'thumb': 'https://m.media-amazon.com/images/I/51CqMDJOODL._AC_SR38,50_.jpg', 'large': 'https://m.media-amazon.com/images/I/51CqMDJOODL._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/91ub366PdKL._AC_UL1500_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/417HN5YFVhL._AC_SR38,50_.jpg', 'large': 'https://m.media-amazon.com/images/I/417HN5YFVhL._AC_.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/61RpcNwx1SL._AC_UL1200_.jpg'}]


In [16]:
import os 
import logging 
import requests 
import pandas as pd 
import requests 
from io import BytesIO 
from PIL import Image 
from ast import literal_eval 
import time

In [28]:
# # config setup 
# BATCH_SIZE= 500
# TIMEOUT=15
# MIN_WIDTH, MIN_HEIGHT=150,150
# OUTPUT_DIR="images"
# LOG_FILE= "craw_log.txt"
# CHECKPOINT_FILE="last_batch.txt"

# # create output file via os
# # create logging file by logging 

# os.makedirs(OUTPUT_DIR, exist_ok=True)
# logging.basicConfig(
#     filename=LOG_FILE,
#     level=logging.INFO,
#     format="%(asctime)s - %(levelname)s - %(message)s"
# )
# # create fake user id 
# def init_session():
#     user_agents = [
#         "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
#         "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)",
#         "Mozilla/5.0 (X11; Linux x86_64)",
#     ]
#     session = requests.Session()
#     session.headers.update({
#         "User-Agent": random.choice(user_agents),
#         "Accept-Language": "en-US,en;q=0.9"
#     })
#     return session

# def parse_product_images(raw):
#     try:
#         if isinstance(raw, str):
#             images = literal_eval(raw)
#         else:
#             images = raw
#     except Exception:
#         return []

#     urls = []
#     for img in images:
#         if isinstance(img, dict) and img.get("variant") == "MAIN":
#             if img.get("hi_res"):
#                 urls.append(img["hi_res"])
#             elif img.get("large"):
#                 urls.append(img["large"])
#             elif img.get("thumb"):
#                 urls.append(img["thumb"])
#     return urls
# def fetch_image_with_retry(session, url, retries=3, delay=3):
#     for attempt in range(retries):
#         try:
#             resp = session.get(url, timeout=TIMEOUT)
#             if resp.status_code == 200:
#                 return resp
#             else:
#                 logging.warning(f"⚠️ HTTP {resp.status_code} for {url}")
#         except Exception as e:
#             logging.warning(f"⚠️ Connection error: {e}")
#         time.sleep(delay + random.uniform(0, 1))
#     return None
# def save_image(content, asin):
#     try:
#         img = Image.open(BytesIO(content)).convert("RGB")
#         w, h = img.size
#         if w < MIN_WIDTH or h < MIN_HEIGHT:
#             logging.warning(f"{asin}: Too small ({w}x{h})")
#             return False
#         path = os.path.join(OUTPUT_DIR, f"{asin}.jpg")
#         img.save(path, format="JPEG", quality=95)
#         return True
#     except Exception as e:
#         logging.error(f"{asin}: Failed to save image -> {e}")
#         return False
# def process_batch(df_batch, batch_id):
#     session = init_session()
#     success, fail = 0, 0

#     for _, row in df_batch.iterrows():
#         asin = row.get("asin", f"item_{_}")
#         urls = parse_product_images(row.get("product_image"))
#         if not urls:
#             logging.warning(f"{asin}: No MAIN image found")
#             fail += 1
#             continue

#         # Tải ảnh
#         url = urls[0]
#         resp = fetch_image_with_retry(session, url)
#         if not resp:
#             logging.error(f"{asin}: Failed after retries")
#             fail += 1
#             continue

#         if save_image(resp.content, asin):
#             success += 1
#             logging.info(f"{asin}: ✅ Saved successfully")
#         else:
#             fail += 1

#         time.sleep(random.uniform(1, 3))  # tránh bị block

#     logging.info(f"Batch {batch_id}: DONE ({success} success, {fail} fail)")
#     return success, fail
# def main(df):
#     # Resume checkpoint nếu có
#     start_idx = 0
#     if os.path.exists(CHECKPOINT_FILE):
#         with open(CHECKPOINT_FILE) as f:
#             start_idx = int(f.read().strip())
#         logging.info(f"Resuming from batch {start_idx}")

#     for i in range(start_idx, len(df), BATCH_SIZE):
#         batch_id = i // BATCH_SIZE + 1
#         df_batch = df.iloc[i:i + BATCH_SIZE]

#         success, fail = process_batch(df_batch, batch_id)
#         logging.info(f"✅ Batch {batch_id} complete: {success} success, {fail} fail")

#         # Ghi checkpoint
#         with open(CHECKPOINT_FILE, "w") as f:
#             f.write(str(i + BATCH_SIZE))

#         # Random cooldown giữa các batch
#         time.sleep(random.uniform(5, 10))

#     logging.info("🎯 All batches completed!")


In [22]:
df_unique = df.drop_duplicates(subset=["asin"]).reset_index(drop=True)
print("Sau khi lọc trùng:", len(df_unique))

Sau khi lọc trùng: 94472


In [23]:
existing_asins = {os.path.splitext(f)[0] for f in os.listdir(OUTPUT_DIR)}
df_unique = df_unique[~df_unique["asin"].isin(existing_asins)].reset_index(drop=True)

In [24]:
print("Sau khi bỏ asin đã có ảnh:", len(df_unique))

Sau khi bỏ asin đã có ảnh: 92894


In [27]:
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

In [33]:
df_unique.to_csv("ready_to_crawl.csv", index=False)

In [ ]:
import os, time, random, logging
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from PIL import Image
from io import BytesIO
import requests
from ast import literal_eval

# ==== CONFIG ====
BATCH_SIZE = 500
TIMEOUT = 15
MIN_WIDTH, MIN_HEIGHT = 150, 150
OUTPUT_DIR = "images"
LOG_FILE = "craw_log.txt"
CHECKPOINT_FILE = "last_batch.txt"
MAX_WORKERS = 5   # số luồng song song
REST_AFTER_BATCHES = 10
REST_DURATION = 420  # 5 phút = 300s

# ==== INIT ====
os.makedirs(OUTPUT_DIR, exist_ok=True)
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ==== SESSION ====
def init_session():
    user_agents = [
        # Windows + Chrome
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        # Windows + Edge
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0",
        # macOS + Safari
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_2) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.2 Safari/605.1.15",
        # macOS + Chrome
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_4_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.5735.198 Safari/537.36",
        # Linux + Firefox
        "Mozilla/5.0 (X11; Linux x86_64; rv:120.0) Gecko/20100101 Firefox/120.0",
        # Android + Chrome
        "Mozilla/5.0 (Linux; Android 13; SM-S918B) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Mobile Safari/537.36",
        # iPhone + Safari
        "Mozilla/5.0 (iPhone; CPU iPhone OS 16_3_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.3 Mobile/15E148 Safari/604.1",
        # iPad + Safari
        "Mozilla/5.0 (iPad; CPU OS 16_3 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/16.3 Mobile/15E148 Safari/604.1",
    ]
    session = requests.Session()
    session.headers.update({
        "User-Agent": random.choice(user_agents),
        "Accept-Language": "en-US,en;q=0.9"
    })
    return session


# ==== IMAGE PARSER ====
def parse_product_images(raw):
    try:
        images = literal_eval(raw) if isinstance(raw, str) else raw
    except Exception:
        return []
    urls = []
    for img in images:
        if isinstance(img, dict) and img.get("variant") == "MAIN":
            for key in ["hi_res", "large", "thumb"]:
                if img.get(key):
                    urls.append(img[key])
                    break
    return urls


# ==== FETCH ====
def fetch_image_with_retry(session, url, retries=3, delay=2):
    for attempt in range(retries):
        try:
            resp = session.get(url, timeout=TIMEOUT)
            if resp.status_code == 200:
                return resp
            else:
                logging.warning(f"⚠️ HTTP {resp.status_code} for {url}")
        except Exception as e:
            logging.warning(f"⚠️ Connection error: {e}")
        time.sleep(delay + random.uniform(1,3))
    return None


# ==== SAVE ====
def save_image(content, asin):
    try:
        img = Image.open(BytesIO(content)).convert("RGB")
        w, h = img.size
        if w < MIN_WIDTH or h < MIN_HEIGHT:
            logging.warning(f"{asin}: Too small ({w}x{h})")
            return False
        path = os.path.join(OUTPUT_DIR, f"{asin}.jpg")
        img.save(path, format="JPEG", quality=95)
        return True
    except Exception as e:
        logging.error(f"{asin}: Failed to save image -> {e}")
        return False


# ==== PROCESS ONE ITEM ====
def process_one(row):
    asin = row.get("asin")
    path = os.path.join(OUTPUT_DIR, f"{asin}.jpg")
    if os.path.exists(path):
        return asin, "skipped"

    urls = parse_product_images(row.get("product_image"))
    if not urls:
        return asin, "no_main"

    session = init_session()
    resp = fetch_image_with_retry(session, urls[0])
    if not resp:
        return asin, "failed"

    if save_image(resp.content, asin):
        return asin, "success"
    return asin, "failed"


# ==== PROCESS BATCH ====
def process_batch(df_batch, batch_id):
    success, fail, skipped = 0, 0, 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_one, row): row for _, row in df_batch.iterrows()}
        for future in as_completed(futures):
            asin, status = future.result()
            if status == "success":
                success += 1
            elif status == "failed":
                fail += 1
            else:
                skipped += 1
            logging.info(f"{asin}: {status}")

    logging.info(f"Batch {batch_id} done ({success} success, {fail} fail, {skipped} skipped)")
    return success, fail, skipped


# ==== MAIN ====
def main(df):
    # # Preprocess: unique & skip existing images
    # df = df.drop_duplicates(subset="asin", keep="first")
    # existing_asins = {os.path.splitext(f)[0] for f in os.listdir(OUTPUT_DIR)}
    # df = df[~df["asin"].isin(existing_asins)]

    start_idx = 0
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            start_idx = int(f.read().strip())
        logging.info(f"Resuming from batch {start_idx}")

    batch_counter = 0
    for i in range(start_idx, len(df), BATCH_SIZE):
        batch_id = i // BATCH_SIZE + 1
        df_batch = df.iloc[i:i + BATCH_SIZE]

        success, fail, skipped = process_batch(df_batch, batch_id)
        logging.info(f"✅ Batch {batch_id} complete: {success} success, {fail} fail, {skipped} skipped")

        with open(CHECKPOINT_FILE, "w") as f:
            f.write(str(i + BATCH_SIZE))

        batch_counter += 1
        if batch_counter % REST_AFTER_BATCHES == 0:
            logging.info(f"😴 Taking a 7-minute rest after {batch_counter} batches...")
            time.sleep(REST_DURATION)
        else:
            time.sleep(random.uniform(10, 15))

    logging.info("🎯 All batches completed!")



In [38]:
main(df_unique)

KeyboardInterrupt: 